# Normalized Update Magnitude Analysis: ||ΔW||_F / ||W||_F

For each (run, layer, module) we compute:
- `deltaW_fro` — Frobenius norm of the low-rank update ΔW
- `W_fro` — Frobenius norm of the corresponding base-model weight
- `deltaW_over_W_fro` — normalized ratio showing how strongly the adapter modifies each layer

This metric is comparable across layers of different dimensionality, unlike raw ||ΔW||_F.

In [ ]:
import os
import re
import json
import warnings
from pathlib import Path

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

sns.set_theme(style="whitegrid", font_scale=0.9)

In [ ]:
RUNS_DIR    = Path("./runs")
MODEL_NAME  = "qwen3_0.6b"
OUTPUT_DIR  = Path("./analysis/qwen3_0.6b")
HEATMAP_DIR = OUTPUT_DIR / "heatmaps_delta_w_norm"
CHARTS_DIR  = OUTPUT_DIR / "charts_delta_w_norm"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
HEATMAP_DIR.mkdir(exist_ok=True)
CHARTS_DIR.mkdir(exist_ok=True)

# Базовая модель для нормировки весов
BASE_MODEL_ID = "Qwen/Qwen3-0.6B"

## 1. Discover runs (reused from rank_analysis)

In [ ]:
def discover_runs(runs_dir: Path) -> list[dict]:
    PATTERNS = [
        re.compile(r"^(?P<dataset>.+?)_lora_r(?P<rank>\d+)$"),
        re.compile(r"^(?P<dataset>.+?)_adalora_init(?P<init_r>\d+)_target(?P<target_r>\d+)$"),
        re.compile(r"^(?P<dataset>.+?)_l1ra_r(?P<rank>\d+)_lam(?P<lam>[\d.e-]+)$"),
    ]
    METHOD_MAP = ["lora", "adalora", "l1ra"]

    runs = []
    for run_path in sorted(runs_dir.iterdir()):
        if not run_path.is_dir():
            continue
        adapter_dir = run_path / "adapter"
        if not adapter_dir.exists():
            warnings.warn(f"No adapter/ in {run_path.name}, skipping")
            continue

        run_name = run_path.name
        meta = {"run_name": run_name, "method": None, "dataset": None, "seed": None, "path": str(adapter_dir)}

        for pat, method in zip(PATTERNS, METHOD_MAP):
            m = pat.match(run_name)
            if m:
                meta["method"] = method
                meta["dataset"] = m.group("dataset")
                break

        if meta["method"] is None:
            warnings.warn(f"Could not parse run_name: {run_name}")
        runs.append(meta)
    return runs


runs = discover_runs(RUNS_DIR)
print(f"Found {len(runs)} runs:")
for r in runs:
    print(f"  {r['method']:>8s} | {r['dataset']:<25s} | {r['run_name']}")

## 2. Load base model weights

In [ ]:
from transformers import AutoModelForCausalLM

print(f"Загрузка базовой модели: {BASE_MODEL_ID} ...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID, torch_dtype=torch.float32,
)
base_state_dict = base_model.state_dict()
print(f"  Загружено параметров: {len(base_state_dict)}")

# Индекс базовых весов: (layer_id, module_name) -> tensor
# Ключи базовой модели: model.layers.{layer}.{self_attn|mlp}.{module}.weight
_BASE_KEY_RE = re.compile(
    r"model\.layers\.(?P<layer>\d+)\.(?P<module>\S+)\.weight$"
)

TARGET_MODULES = {"self_attn.q_proj", "self_attn.k_proj", "self_attn.v_proj",
                  "self_attn.o_proj", "mlp.gate_proj", "mlp.up_proj", "mlp.down_proj"}

base_weights: dict[tuple[int, str], torch.Tensor] = {}
for key, tensor in base_state_dict.items():
    m = _BASE_KEY_RE.match(key)
    if m is None:
        continue
    module_name = m.group("module")
    if module_name not in TARGET_MODULES:
        continue
    layer_id = int(m.group("layer"))
    base_weights[(layer_id, module_name)] = tensor.float()

print(f"  Проиндексировано матриц: {len(base_weights)}")

# Полная модель больше не нужна, оставляем только веса
del base_model
import gc; gc.collect()

## 3. Helpers: adapter loading, ΔW computation, energy rank, L1RA pruning

In [ ]:
def _load_state_dict(adapter_dir: str) -> dict[str, torch.Tensor]:
    adapter_path = Path(adapter_dir)
    st_file = adapter_path / "adapter_model.safetensors"
    pt_file = adapter_path / "adapter_model.pt"
    if st_file.exists():
        from safetensors.torch import load_file
        return load_file(str(st_file), device="cpu")
    elif pt_file.exists():
        return torch.load(str(pt_file), map_location="cpu", weights_only=True)
    else:
        raise FileNotFoundError(f"Веса адаптера не найдены: {adapter_dir}")


_KEY_PATTERNS = [
    re.compile(r"layers\.(?P<layer>\d+)\.(?P<module>\S+?)\.(?P<comp>lora_[ABc]|lora_E)\.weight$"),
    re.compile(r"layers\.(?P<layer>\d+)\.(?P<module>\S+?)\.(?P<comp>lora_[ABE])$"),
    re.compile(r"layers\.(?P<layer>\d+)\.(?P<module>\S+?)\.(?P<comp>lora_[ABc])\.default$"),
]


def _parse_key(key: str):
    for pat in _KEY_PATTERNS:
        m = pat.search(key)
        if m:
            return int(m.group("layer")), m.group("module"), m.group("comp")
    return None


def extract_delta_w(state_dict: dict[str, torch.Tensor], method: str) -> list[dict]:
    """Group keys by (layer, module) and compute ΔW.
    For lora/adalora — standard computation.
    For l1ra — returns raw (unpruned) ΔW. Use extract_l1ra_pruned_delta_w for pruned version.
    """
    groups: dict[tuple, dict] = {}
    for key, tensor in state_dict.items():
        parsed = _parse_key(key)
        if parsed is None:
            continue
        layer_id, module_name, comp = parsed
        groups.setdefault((layer_id, module_name), {})[comp] = tensor

    results = []
    for (layer_id, module_name), tensors in sorted(groups.items()):
        A = tensors.get("lora_A")
        B = tensors.get("lora_B")
        if A is None or B is None:
            continue
        A, B = A.float(), B.float()

        if A.numel() == 0 or B.numel() == 0:
            results.append({"layer_id": layer_id, "module_name": module_name,
                            "nominal_rank": 0, "delta_w": None})
            continue

        if method == "adalora":
            E = tensors.get("lora_E")
            if E is not None:
                E = E.float().flatten()  # flatten to 1D (squeeze can produce 0D on [1,1])
                if E.numel() == 0:
                    results.append({"layer_id": layer_id, "module_name": module_name,
                                    "nominal_rank": 0, "delta_w": None})
                    continue
                delta_w = B @ torch.diag(E) @ A
            else:
                delta_w = B @ A
            nominal_rank = A.shape[0]
        elif method == "l1ra":
            c = tensors.get("lora_c")
            if c is not None:
                c = c.float().flatten()
                delta_w = (A * c) @ B
            else:
                delta_w = A @ B
            nominal_rank = A.shape[1]
        else:  # lora
            delta_w = B @ A
            nominal_rank = A.shape[0]

        results.append({"layer_id": layer_id, "module_name": module_name,
                        "nominal_rank": nominal_rank, "delta_w": delta_w})
    return results


# ---------------------------------------------------------------------------
# L1RA pruning helpers
# ---------------------------------------------------------------------------

def extract_l1ra_components(state_dict: dict[str, torch.Tensor]) -> list[dict]:
    """Extract (A, B, c) triples for each (layer, module) from L1RA adapter."""
    groups: dict[tuple, dict] = {}
    for key, tensor in state_dict.items():
        parsed = _parse_key(key)
        if parsed is None:
            continue
        layer_id, module_name, comp = parsed
        groups.setdefault((layer_id, module_name), {})[comp] = tensor.float()

    results = []
    for (layer_id, module_name), tensors in sorted(groups.items()):
        A = tensors.get("lora_A")
        B = tensors.get("lora_B")
        c = tensors.get("lora_c")
        if A is None or B is None or c is None:
            continue
        results.append({"layer_id": layer_id, "module_name": module_name,
                        "A": A, "B": B, "c": c.flatten()})
    return results


def find_threshold_for_target_active_rank(
    components: list[dict], target_rank: float = 32.0,
    tol: float = 0.3, max_iter: int = 60,
) -> dict:
    """Binary search for threshold τ such that mean #{|c_i| > τ} ≈ target_rank."""
    all_c_abs = torch.cat([comp["c"].abs() for comp in components])
    lo, hi = 0.0, float(all_c_abs.max().item())
    best = {"threshold": 0.0, "mean_active_rank": None, "gap": float("inf")}

    for _ in range(max_iter):
        mid = (lo + hi) / 2
        ranks = [int((comp["c"].abs() > mid).sum().item()) for comp in components]
        mean_ar = float(np.mean(ranks))
        gap = abs(mean_ar - target_rank)
        if gap < best["gap"]:
            best = {"threshold": mid, "mean_active_rank": mean_ar, "gap": gap}
        if gap < tol:
            break
        if mean_ar > target_rank:
            lo = mid
        else:
            hi = mid
    return best


def extract_l1ra_pruned_delta_w(
    components: list[dict], threshold: float
) -> list[dict]:
    """Compute pruned ΔW for L1RA: zero out c_i where |c_i| <= threshold."""
    results = []
    for comp in components:
        c = comp["c"]
        mask = c.abs() > threshold
        active_r = int(mask.sum().item())

        if active_r == 0:
            results.append({"layer_id": comp["layer_id"], "module_name": comp["module_name"],
                            "nominal_rank": int(c.numel()), "active_rank": 0, "delta_w": None})
            continue

        c_masked = c.clone()
        c_masked[~mask] = 0.0
        delta_w = (comp["A"] * c_masked) @ comp["B"]
        results.append({"layer_id": comp["layer_id"], "module_name": comp["module_name"],
                        "nominal_rank": int(c.numel()), "active_rank": active_r, "delta_w": delta_w})
    return results


# ---------------------------------------------------------------------------
# Shared helpers
# ---------------------------------------------------------------------------

def energy_rank(delta_w: torch.Tensor, threshold: float = 0.95) -> int:
    if delta_w is None or delta_w.numel() == 0:
        return 0
    try:
        sigma = torch.linalg.svdvals(delta_w)
    except Exception:
        return 0
    energy = sigma ** 2
    total = energy.sum().item()
    if total < 1e-12:
        return 0
    cumulative = torch.cumsum(energy, dim=0) / total
    k = int((cumulative >= threshold).nonzero(as_tuple=True)[0][0].item()) + 1
    return k


def short_module_name(name: str) -> str:
    return name.split(".")[-1]


MODULE_ORDER = ["q_proj", "k_proj", "v_proj", "o_proj",
                "gate_proj", "up_proj", "down_proj"]

ATTN_MODULES = {"q_proj", "k_proj", "v_proj", "o_proj"}
MLP_MODULES  = {"gate_proj", "up_proj", "down_proj"}

def module_group(name: str) -> str:
    s = short_module_name(name)
    if s in ATTN_MODULES:
        return "attention"
    elif s in MLP_MODULES:
        return "MLP"
    return "other"

L1RA_TARGET_RANK = 32.0

## 4. Main loop — compute deltaW_over_W_fro for all runs

**Fair comparison protocol:**
- LoRA: r = 32 (fixed)
- AdaLoRA: target_rank ≈ 32 (SVD-based pruning during training)
- L1RA: threshold pruning on `lora_c` so that mean active_rank ≈ 32

In [ ]:
all_rows = []
failed_runs = []
missing_base_weights = []
l1ra_threshold_info = {}  # run_name -> {threshold, mean_active_rank_before, mean_active_rank_after}

for run_meta in runs:
    run_name = run_meta["run_name"]
    method = run_meta["method"]
    print(f"\nОбработка: {run_name} ...", end=" ")

    try:
        state_dict = _load_state_dict(run_meta["path"])

        # ------------------------------------------------------------------
        # Для L1RA сначала подбираем threshold, затем считаем pruned ΔW
        # ------------------------------------------------------------------
        if method == "l1ra":
            components = extract_l1ra_components(state_dict)
            # Средний активный ранг до pruning
            raw_ranks = [int(comp["c"].numel()) for comp in components]
            mean_ar_before = float(np.mean(raw_ranks))

            # Подбор threshold τ
            res = find_threshold_for_target_active_rank(
                components, target_rank=L1RA_TARGET_RANK
            )
            threshold = res["threshold"]
            mean_ar_after = res["mean_active_rank"]
            print(f"[L1RA] τ={threshold:.6f}, active_rank: {mean_ar_before:.0f} → {mean_ar_after:.1f}")

            l1ra_threshold_info[run_name] = {
                "threshold": threshold,
                "mean_active_rank_before": mean_ar_before,
                "mean_active_rank_after": mean_ar_after,
            }

            # Расчет pruned ΔW
            modules = extract_l1ra_pruned_delta_w(components, threshold)
            del components
        else:
            modules = extract_delta_w(state_dict, method)

        # ------------------------------------------------------------------
        # Метрики для каждой пары (layer, module)
        # ------------------------------------------------------------------
        for mod in tqdm(modules, desc=run_name, leave=False):
            layer_id = mod["layer_id"]
            module_name = mod["module_name"]
            delta_w = mod["delta_w"]

            e_rank_95 = energy_rank(delta_w, threshold=0.95)
            e_rank_99 = energy_rank(delta_w, threshold=0.99)

            # Норма ΔW
            if delta_w is not None and delta_w.numel() > 0:
                dw_fro = delta_w.norm().item()
            else:
                dw_fro = 0.0

            # Норма базового веса
            W = base_weights.get((layer_id, module_name))
            if W is not None:
                w_fro = W.norm().item()
                ratio = dw_fro / w_fro if w_fro > 1e-12 else float("nan")
            else:
                w_fro = float("nan")
                ratio = float("nan")
                missing_base_weights.append((run_name, layer_id, module_name))

            row = {
                "run_name":           run_name,
                "method":             method,
                "dataset":            run_meta["dataset"],
                "layer_id":           layer_id,
                "module_name":        module_name,
                "nominal_rank":       mod["nominal_rank"],
                "energy_rank_95":     e_rank_95,
                "energy_rank_99":     e_rank_99,
                "deltaW_fro":         dw_fro,
                "W_fro":              w_fro,
                "deltaW_over_W_fro":  ratio,
            }
            # For L1RA add active_rank
            if method == "l1ra":
                row["active_rank"] = mod.get("active_rank", mod["nominal_rank"])
            all_rows.append(row)

        print(f"OK ({len(modules)} modules)")
        del state_dict, modules

    except Exception as e:
        print(f"ошибка: {e}")
        import traceback; traceback.print_exc()
        failed_runs.append({"run_name": run_name, "error": str(e)})

print(f"\n{'='*60}")
print(f"Processed: {len(runs) - len(failed_runs)}/{len(runs)} runs")
if failed_runs:
    print("\nFailed runs:")
    for f in failed_runs:
        print(f"  {f['run_name']}: {f['error']}")
if missing_base_weights:
    print(f"\nMissing base weights for {len(missing_base_weights)} (run, layer, module) pairs")
if l1ra_threshold_info:
    print("\nL1RA threshold calibration:")
    for rn, info in l1ra_threshold_info.items():
        print(f"  {rn}: τ={info['threshold']:.6f}, "
              f"active_rank {info['mean_active_rank_before']:.0f} → {info['mean_active_rank_after']:.1f}")

## 5. Save layer-wise results

In [ ]:
df = pd.DataFrame(all_rows)

# Fill active_rank for non-L1RA methods (not applicable)
if "active_rank" not in df.columns:
    df["active_rank"] = np.nan

# Derived columns
df["module_short"] = df["module_name"].apply(short_module_name)
df["module_group"] = df["module_name"].apply(module_group)

csv_path = OUTPUT_DIR / "delta_w_norm_analysis.csv"
parquet_path = OUTPUT_DIR / "delta_w_norm_analysis.parquet"
df.to_csv(csv_path, index=False)
df.to_parquet(parquet_path, index=False)

print(f"Saved {len(df)} rows  ({df['run_name'].nunique()} runs × {df['module_name'].nunique()} modules × layers)")
print(f"  CSV:     {csv_path}")
print(f"  Parquet: {parquet_path}")
print(f"\nRuns: {sorted(df['run_name'].unique())}")
print(f"Methods: {sorted(df['method'].unique())}")
print(f"Datasets: {sorted(df['dataset'].unique())}")
df.head(10)

## 6. Heatmaps per run: energy_rank_95 | energy_rank_99 | deltaW_over_W_fro (side-by-side)

For L1RA runs, these are the **pruned** metrics (after threshold calibration to active_rank ≈ 32).

In [ ]:
for run_name, group in df.groupby("run_name"):
    g = group.copy()
    method = g["method"].iloc[0]
    dataset = g["dataset"].iloc[0]

    # Per-dataset subdirectory
    ds_dir = HEATMAP_DIR / dataset
    ds_dir.mkdir(exist_ok=True)

    # --- Side-by-side: energy_rank_95 | energy_rank_99 | deltaW_over_W_fro ---
    fig, axes = plt.subplots(1, 3, figsize=(38, 4), sharey=True)

    # Left: energy_rank_95
    pivot_er95 = g.pivot_table(index="module_short", columns="layer_id",
                               values="energy_rank_95", aggfunc="first")
    pivot_er95 = pivot_er95.reindex([m for m in MODULE_ORDER if m in pivot_er95.index])
    sns.heatmap(pivot_er95, annot=True, fmt=".0f", cmap="YlOrRd",
                vmin=0, vmax=32, linewidths=0.5, ax=axes[0])
    axes[0].set_title(f"energy_rank_95  (mean={g['energy_rank_95'].mean():.1f})", fontsize=10)
    axes[0].set_xlabel("Layer")
    axes[0].set_ylabel("")

    # Middle: energy_rank_99
    pivot_er99 = g.pivot_table(index="module_short", columns="layer_id",
                               values="energy_rank_99", aggfunc="first")
    pivot_er99 = pivot_er99.reindex([m for m in MODULE_ORDER if m in pivot_er99.index])
    sns.heatmap(pivot_er99, annot=True, fmt=".0f", cmap="YlOrRd",
                vmin=0, vmax=48, linewidths=0.5, ax=axes[1])
    axes[1].set_title(f"energy_rank_99  (mean={g['energy_rank_99'].mean():.1f})", fontsize=10)
    axes[1].set_xlabel("Layer")

    # Right: deltaW_over_W_fro
    pivot_dw = g.pivot_table(index="module_short", columns="layer_id",
                             values="deltaW_over_W_fro", aggfunc="first")
    pivot_dw = pivot_dw.reindex([m for m in MODULE_ORDER if m in pivot_dw.index])
    vmax_dw = pivot_dw.max().max()
    sns.heatmap(pivot_dw, annot=True, fmt=".3f", cmap="YlOrRd",
                vmin=0, vmax=vmax_dw, linewidths=0.5, ax=axes[2])
    axes[2].set_title(f"||ΔW||_F / ||W||_F  (mean={g['deltaW_over_W_fro'].mean():.4f})", fontsize=10)
    axes[2].set_xlabel("Layer")

    label = f"{method.upper()} / {dataset}"
    if method == "l1ra":
        info = l1ra_threshold_info.get(run_name, {})
        label += f"  [pruned: active_rank≈{info.get('mean_active_rank_after', '?'):.0f}]"
    fig.suptitle(label, fontsize=12, y=1.03)
    fig.tight_layout()
    fig_path = ds_dir / f"{method}_side_by_side.png"
    fig.savefig(fig_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fig_path}")

## 7. Summary tables

In [ ]:
# 7a. Per-module summary (averaged across layers)
print("=" * 80)
print("Summary per module (L1RA = pruned to active_rank ≈ 32)")
print("=" * 80)

module_summary = (
    df.groupby(["method", "dataset", "module_short"])
    .agg(
        avg_energy_rank_95=("energy_rank_95", "mean"),
        avg_energy_rank_99=("energy_rank_99", "mean"),
        avg_deltaW_over_W_fro=("deltaW_over_W_fro", "mean"),
        avg_deltaW_fro=("deltaW_fro", "mean"),
    )
    .reset_index()
    .sort_values(["dataset", "method", "module_short"])
    .reset_index(drop=True)
)
display(module_summary.round(4))

In [ ]:
# 7b. Attention vs MLP group summary
print("=" * 80)
print("Summary: attention vs MLP (L1RA = pruned)")
print("=" * 80)

group_summary = (
    df.groupby(["method", "dataset", "module_group"])
    .agg(
        avg_energy_rank_95=("energy_rank_95", "mean"),
        avg_energy_rank_99=("energy_rank_99", "mean"),
        avg_deltaW_over_W_fro=("deltaW_over_W_fro", "mean"),
        avg_deltaW_fro=("deltaW_fro", "mean"),
        n_modules=("energy_rank_95", "count"),
    )
    .reset_index()
    .sort_values(["dataset", "method", "module_group"])
    .reset_index(drop=True)
)
display(group_summary.round(4))

# Compact cross-table: method × dataset, MLP/attention ratio of deltaW_over_W_fro
print("\n" + "=" * 80)
print("MLP / attention ratio of avg ||ΔW||_F/||W||_F")
print("=" * 80)
pivot = group_summary.pivot_table(
    index=["method", "dataset"], columns="module_group",
    values="avg_deltaW_over_W_fro"
)
pivot["MLP_over_attn"] = pivot["MLP"] / pivot["attention"]
display(pivot.round(4))

# Same for energy_rank_99
print("\n" + "=" * 80)
print("MLP / attention ratio of avg energy_rank_99")
print("=" * 80)
pivot_er99 = group_summary.pivot_table(
    index=["method", "dataset"], columns="module_group",
    values="avg_energy_rank_99"
)
pivot_er99["MLP_over_attn"] = pivot_er99["MLP"] / pivot_er99["attention"]
display(pivot_er99.round(4))

In [ ]:
# 7c. Overall run-level summary
print("=" * 80)
print("Overall run-level summary (L1RA = pruned)")
print("=" * 80)

run_summary = (
    df.groupby(["run_name", "method", "dataset"])
    .agg(
        nominal_rank=("nominal_rank", "max"),
        mean_energy_rank_95=("energy_rank_95", "mean"),
        median_energy_rank_95=("energy_rank_95", "median"),
        mean_energy_rank_99=("energy_rank_99", "mean"),
        median_energy_rank_99=("energy_rank_99", "median"),
        mean_deltaW_over_W_fro=("deltaW_over_W_fro", "mean"),
        median_deltaW_over_W_fro=("deltaW_over_W_fro", "median"),
        mean_deltaW_fro=("deltaW_fro", "mean"),
        n_modules=("energy_rank_95", "count"),
    )
    .reset_index()
    .sort_values(["dataset", "method"])
    .reset_index(drop=True)
)
display(run_summary.round(4))

## 8. Cross-method comparison charts

Bar charts comparing `||ΔW||_F / ||W||_F` across methods, grouped by module and dataset.
All methods at comparable effective rank ≈ 32.

In [ ]:
# 8a. Bar chart: avg deltaW_over_W_fro per module, faceted by dataset
datasets = sorted(df["dataset"].unique())
methods = sorted(df["method"].unique())
n_ds = len(datasets)

fig, axes = plt.subplots(1, n_ds, figsize=(7 * n_ds, 5), sharey=True)
if n_ds == 1:
    axes = [axes]

for ax, ds in zip(axes, datasets):
    subset = module_summary[module_summary["dataset"] == ds]
    pivot = subset.pivot(index="module_short", columns="method", values="avg_deltaW_over_W_fro")
    pivot = pivot.reindex([m for m in MODULE_ORDER if m in pivot.index])
    pivot.plot(kind="bar", ax=ax, width=0.75)
    ax.set_title(ds, fontsize=12)
    ax.set_ylabel("avg ||ΔW||_F / ||W||_F" if ax == axes[0] else "")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=45)
    ax.legend(title="method")

fig.suptitle("Normalized update magnitude per module (all methods at budget ≈ 32)", fontsize=13, y=1.02)
fig.tight_layout()
p = CHARTS_DIR / "cross_method_deltaW_over_W_by_module.png"
fig.savefig(p, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {p}")

# 8a-2. Bar chart: avg energy_rank_99 per module, faceted by dataset
fig, axes = plt.subplots(1, n_ds, figsize=(7 * n_ds, 5), sharey=True)
if n_ds == 1:
    axes = [axes]

for ax, ds in zip(axes, datasets):
    subset = module_summary[module_summary["dataset"] == ds]
    pivot = subset.pivot(index="module_short", columns="method", values="avg_energy_rank_99")
    pivot = pivot.reindex([m for m in MODULE_ORDER if m in pivot.index])
    pivot.plot(kind="bar", ax=ax, width=0.75)
    ax.set_title(ds, fontsize=12)
    ax.set_ylabel("avg energy_rank_99" if ax == axes[0] else "")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=45)
    ax.legend(title="method")

fig.suptitle("Energy rank (99%) per module (all methods at budget ≈ 32)", fontsize=13, y=1.02)
fig.tight_layout()
p = CHARTS_DIR / "cross_method_energy_rank_99_by_module.png"
fig.savefig(p, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {p}")

In [ ]:
# 8b. Attention vs MLP grouped bar chart — deltaW_over_W_fro
fig, axes = plt.subplots(1, n_ds, figsize=(6 * n_ds, 4.5), sharey=True)
if n_ds == 1:
    axes = [axes]

for ax, ds in zip(axes, datasets):
    subset = group_summary[group_summary["dataset"] == ds]
    pivot = subset.pivot(index="module_group", columns="method", values="avg_deltaW_over_W_fro")
    pivot = pivot.reindex(["attention", "MLP"])
    pivot.plot(kind="bar", ax=ax, width=0.6)
    ax.set_title(ds, fontsize=12)
    ax.set_ylabel("avg ||ΔW||_F / ||W||_F" if ax == axes[0] else "")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=0)
    ax.legend(title="method")

fig.suptitle("Attention vs MLP: normalized update magnitude (budget ≈ 32)", fontsize=13, y=1.02)
fig.tight_layout()
p = CHARTS_DIR / "cross_method_attn_vs_mlp.png"
fig.savefig(p, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {p}")

# 8b-2. Attention vs MLP grouped bar chart — energy_rank_99
fig, axes = plt.subplots(1, n_ds, figsize=(6 * n_ds, 4.5), sharey=True)
if n_ds == 1:
    axes = [axes]

for ax, ds in zip(axes, datasets):
    subset = group_summary[group_summary["dataset"] == ds]
    pivot = subset.pivot(index="module_group", columns="method", values="avg_energy_rank_99")
    pivot = pivot.reindex(["attention", "MLP"])
    pivot.plot(kind="bar", ax=ax, width=0.6)
    ax.set_title(ds, fontsize=12)
    ax.set_ylabel("avg energy_rank_99" if ax == axes[0] else "")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=0)
    ax.legend(title="method")

fig.suptitle("Attention vs MLP: energy rank 99% (budget ≈ 32)", fontsize=13, y=1.02)
fig.tight_layout()
p = CHARTS_DIR / "cross_method_attn_vs_mlp_er99.png"
fig.savefig(p, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {p}")

## 9. Interpretation guide

### What to look for in the results

1. **Which modules have the highest `deltaW_over_W_fro`?**
   - If `o_proj`, `down_proj` consistently show larger relative updates — this is genuine, not a dimension artifact.

2. **Does the pattern align with `energy_rank_95`?**
   - High norm + high rank → complex, high-dimensional adaptation.
   - High norm + low rank → strong but low-dimensional (scaling-like) update.

3. **Attention vs MLP**
   - If MLP > attention in normalized magnitude across all 3 datasets — supports the hypothesis that MLP layers are more heavily adapted.

4. **Cross-dataset stability**
   - Do the same modules dominate across meta_math, open_code_instruct, open_orca?
   - If yes → the pattern reflects model architecture, not task specificity.

5. **Method comparison at budget ≈ 32**
   - AdaLoRA and L1RA should show more heterogeneous layer-wise profiles than LoRA.
   - The key question: do adaptive methods concentrate updates in the same modules?